## 🌦️ External Data ETL Pipeline: Weather Data Extraction

This notebook is part of an **ETL (Extract, Transform, Load) pipeline** that collects historical weather data from the Open-Meteo API for multiple global cities.

---

### 📌 Objective
The goal of this project is to:
- Extract historical weather data for major cities
- Transform raw API responses into a clean structured format
- Prepare a time-series dataset for analysis and downstream use cases

---

### 🌍 Cities Covered
- New York
- Los Angeles
- Chicago
- Hamburg
- Shanghai
- Mumbai

---

### 🛠️ Final Output
A cleaned and consolidated weather dataset containing daily observations such as:
- Temperature (max, min, mean)
- Precipitation, rain, snowfall
- Wind speed
- Weather condition codes

This dataset will be saved as a CSV file for further analysis and modeling.

## 1. Install Required Libraries

This step installs all necessary Python libraries required for fetching and processing weather data from the Open-Meteo API.

We use:
- `openmeteo-requests` → to access Open-Meteo API
- `requests-cache` → to cache API responses and avoid repeated calls
- `retry-requests` → to handle request failures with automatic retries
- `numpy` and `pandas` → for data manipulation and analysis

In [2]:
%pip install openmeteo-requests
%pip install requests-cache retry-requests numpy pandas

  Using cached openmeteo_requests-1.7.5-py3-none-any.whl.metadata (11 kB)
  Using cached niquests-3.20.0-py3-none-any.whl.metadata (16 kB)
  Using cached openmeteo_sdk-1.27.2-py3-none-any.whl.metadata (935 bytes)
  Using cached urllib3_future-2.22.900-py3-none-any.whl.metadata (20 kB)
  Using cached wassima-2.1.1-py3-none-any.whl.metadata (5.6 kB)
  Using cached jh2-5.0.13-cp37-abi3-win_amd64.whl.metadata (4.1 kB)
  Using cached qh3-1.9.2-cp37-abi3-win_amd64.whl.metadata (5.2 kB)
  Using cached flatbuffers-25.9.23-py2.py3-none-any.whl.metadata (875 bytes)
Using cached openmeteo_requests-1.7.5-py3-none-any.whl (7.1 kB)
Using cached niquests-3.20.0-py3-none-any.whl (213 kB)
Using cached urllib3_future-2.22.900-py3-none-any.whl (775 kB)
Using cached jh2-5.0.13-cp37-abi3-win_amd64.whl (252 kB)
Using cached qh3-1.9.2-cp37-abi3-win_amd64.whl (2.1 MB)
Using cached wassima-2.1.1-py3-none-any.whl (131 kB)
Using cached openmeteo_sdk-1.27.2-py3-none-any.whl (19 kB)
Using cached flatbuffers-25.9.2

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
mediapipe 0.10.14 requires protobuf<5,>=4.25.3, but you have protobuf 5.29.4 which is incompatible.


  Using cached retry_requests-2.0.0-py3-none-any.whl.metadata (2.6 kB)
  Using cached cattrs-26.1.0-py3-none-any.whl.metadata (8.5 kB)
  Using cached url_normalize-3.0.0-py3-none-any.whl.metadata (7.5 kB)
  Using cached attrs-26.1.0-py3-none-any.whl.metadata (8.8 kB)
Using cached retry_requests-2.0.0-py3-none-any.whl (15 kB)
Using cached cattrs-26.1.0-py3-none-any.whl (73 kB)
Using cached attrs-26.1.0-py3-none-any.whl (67 kB)
Using cached url_normalize-3.0.0-py3-none-any.whl (16 kB)

   ---------------------------------------- 0/6 [url-normalize]
   ---------------------------------------- 0/6 [url-normalize]
  Attempting uninstall: typing-extensions
   ---------------------------------------- 0/6 [url-normalize]
   ------ --------------------------------- 1/6 [typing-extensions]
    Found existing installation: typing_extensions 4.13.0
   ------ --------------------------------- 1/6 [typing-extensions]
    Uninstalling typing_extensions-4.13.0:
   ------ ------------------------------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
mediapipe 0.10.14 requires protobuf<5,>=4.25.3, but you have protobuf 5.29.4 which is incompatible.



---


## 2. Import Libraries and Configure API Client

In this step, we import all required libraries and configure the Open-Meteo API client.

We also set up:
- A **cached session** to store API responses locally
- A **retry mechanism** to handle temporary network failures
- The Open-Meteo client using the configured session

This ensures efficient and reliable data retrieval.

In [6]:
import openmeteo_requests

import pandas as pd

import requests_cache

from retry_requests import retry


---

## 3. Define API Request Parameters

Here we define the API endpoint and request parameters.

We specify:
- Latitude and longitude of multiple cities
- Time range (2015–2018)
- Weather variables such as temperature, precipitation, snowfall, rain, wind speed, and weather codes
- Timezone as GMT

These parameters control what weather data will be retrieved from the API.

In [7]:
# Setup the Open-Meteo API client with cache and retry
cache_session = requests_cache.CachedSession(
    ".cache",
    expire_after=-1
)

retry_session = retry(
    cache_session,
    retries=5,
    backoff_factor=0.2
)

openmeteo = openmeteo_requests.Client(
    session=retry_session
)


---

## 4. Fetch Weather Data from Open-Meteo API

In this step, we send a request to the Open-Meteo archive API using the defined parameters.

The API returns daily weather data for each city, including:
- Temperature (max, min, mean)
- Precipitation and rainfall
- Snowfall
- Wind speed
- Weather condition codes

In [25]:
url = "https://archive-api.open-meteo.com/v1/archive"

params = {
    "latitude": [
        40.7143,    # New York
        34.0522,    # Los Angeles
        41.8500,    # Chicago
        53.5511,    # Hamburg
        31.2304,    # Shanghai
        19.0760     # Mumbai
    ],

    "longitude": [
        -74.0060,
        -118.2437,
        -87.6500,
        9.9937,
        121.4737,
        72.8777
    ],

    "start_date": "2015-01-01",

    "end_date": "2018-12-31",

    "daily": [
        "temperature_2m_max",
        "temperature_2m_min",
        "temperature_2m_mean",
        "precipitation_sum",
        "snowfall_sum",
        "rain_sum",
        "wind_speed_10m_max",
        "weather_code"
    ],

    "timezone": "GMT",
}

responses = openmeteo.weather_api(url, params=params)


---

## 5. Convert API Response into Structured Data

Here, we process the API response for each city and convert it into a structured Pandas DataFrame.

For each city:
- Extract daily weather variables
- Create a date range from start to end date
- Store all weather metrics in a tabular format
- Add a city column for identification

Finally, all city-level data is combined into a single dataset.

In [37]:
city_names = [
    "New York",
    "Los Angeles",
    "Chicago",
    "Hamburg",
    "Shanghai",
    "Mumbai"
]

weather_data = []

for city, response in zip(city_names, responses):

    daily = response.Daily()

    daily_dataframe = pd.DataFrame({

    "date": pd.date_range(
        start="2015-01-01",
        end="2018-12-31",
        freq="D"
    ),

        "temperature_2m_max":
            daily.Variables(0).ValuesAsNumpy(),

        "temperature_2m_min":
            daily.Variables(1).ValuesAsNumpy(),

        "temperature_2m_mean":
            daily.Variables(2).ValuesAsNumpy(),

        "precipitation_sum":
            daily.Variables(3).ValuesAsNumpy(),

        "snowfall_sum":
            daily.Variables(4).ValuesAsNumpy(),

        "rain_sum":
            daily.Variables(5).ValuesAsNumpy(),

        "wind_speed_10m_max":
            daily.Variables(6).ValuesAsNumpy(),

        "weather_code":
            daily.Variables(7).ValuesAsNumpy()
    })

    daily_dataframe["city"] = city

    weather_data.append(daily_dataframe)


---

## 6. Combine and Structure Final Dataset

All individual city DataFrames are merged into a single DataFrame using `pd.concat()`.

We then:
- Standardize column order for consistency
- Ensure the dataset is ready for analysis and downstream processing

In [38]:
weather = pd.concat(
    weather_data,
    ignore_index=True
)

In [39]:
weather = weather[
    [
        "date",
        "city",
        "temperature_2m_max",
        "temperature_2m_min",
        "temperature_2m_mean",
        "precipitation_sum",
        "snowfall_sum",
        "rain_sum",
        "wind_speed_10m_max",
        "weather_code"
    ]
]


---

## 7. Initial Data Exploration

We perform basic exploratory data analysis (EDA) to understand the dataset structure.

This includes:
- Viewing first and last rows (`head`, `tail`)
- Checking data types and null values (`info`, `dtypes`)
- Aggregating date ranges per city

This helps validate that the data has been correctly loaded and structured.

In [40]:
weather.head()

,date,city,temperature_2m_max,temperature_2m_min,temperature_2m_mean,precipitation_sum,snowfall_sum,rain_sum,wind_speed_10m_max,weather_code
0,2015-01-01,New York,2.2835,-4.5665,-2.281083,0.0,0.00,0.0,21.267441,1.0
1,2015-01-02,New York,5.1335,-0.8165,1.043917,0.0,0.00,0.0,19.881649,3.0
2,2015-01-03,New York,3.6835,-3.4165,-0.447750,10.2,3.08,5.8,15.175612,75.0
3,2015-01-04,New York,13.2335,4.4335,8.625167,11.1,0.00,11.1,19.969458,61.0
4,2015-01-05,New York,11.9835,-1.6165,3.350167,2.4,0.00,2.4,26.756500,55.0


In [41]:
weather.tail()

,date,city,temperature_2m_max,temperature_2m_min,temperature_2m_mean,precipitation_sum,snowfall_sum,rain_sum,wind_speed_10m_max,weather_code
8761,2018-12-27,Mumbai,29.000000,17.049999,21.760416,0.0,0.0,0.0,18.455048,2.0
8762,2018-12-28,Mumbai,29.250000,16.450001,21.479164,0.0,0.0,0.0,19.296135,3.0
8763,2018-12-29,Mumbai,29.650000,15.750000,21.900002,0.0,0.0,0.0,17.902534,3.0
8764,2018-12-30,Mumbai,30.600000,15.400000,22.018751,0.0,0.0,0.0,14.587776,1.0
8765,2018-12-31,Mumbai,30.450001,15.350000,22.177080,0.0,0.0,0.0,13.896187,3.0


In [42]:
weather.info()

<class 'pandas.DataFrame'>
RangeIndex: 8766 entries, 0 to 8765
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   date                 8766 non-null   datetime64[us]
 1   city                 8766 non-null   str           
 2   temperature_2m_max   8766 non-null   float32       
 3   temperature_2m_min   8766 non-null   float32       
 4   temperature_2m_mean  8766 non-null   float32       
 5   precipitation_sum    8766 non-null   float32       
 6   snowfall_sum         8766 non-null   float32       
 7   rain_sum             8766 non-null   float32       
 8   wind_speed_10m_max   8766 non-null   float32       
 9   weather_code         8766 non-null   float32       
dtypes: datetime64[us](1), float32(8), str(1)
memory usage: 411.0 KB



---

## 8. Data Type Conversion

The `date` column is converted into a proper datetime format to ensure correct time-series analysis.

This allows:
- Easier filtering by date
- Proper time-based grouping and visualization
- Compatibility with time-series models

In [43]:
weather["date"].dtype

dtype('<M8[us]')

In [44]:
weather.groupby("city")["date"].agg(["min", "max"])

,min,max
city,,
Chicago,2015-01-01,2018-12-31
Hamburg,2015-01-01,2018-12-31
Los Angeles,2015-01-01,2018-12-31
Mumbai,2015-01-01,2018-12-31
New York,2015-01-01,2018-12-31
Shanghai,2015-01-01,2018-12-31


In [45]:
weather["date"] = pd.to_datetime(
    weather["date"].apply(lambda x: x.date())
)

In [46]:
weather.dtypes

date                   datetime64[s]
city                             str
temperature_2m_max           float32
temperature_2m_min           float32
temperature_2m_mean          float32
precipitation_sum            float32
snowfall_sum                 float32
rain_sum                     float32
wind_speed_10m_max           float32
weather_code                 float32
dtype: object


---

## 9. Save Cleaned Dataset

Finally, the processed weather dataset is saved as a CSV file for future use.

This file can be used for:
- Data analysis
- Visualization
- Machine learning models
- Integration with other datasets

A confirmation message is printed after successful saving.

In [48]:
weather.to_csv(
    "../Datasets/External Datasets/weather.csv",
    index=False
)

print("✅ Cleaned weather data saved.")

✅ Cleaned weather data saved.
